# TP Jour 4 — Tests d'Hypothèses Statistiques, Segmentation & Restitution BI

### Objectifs de la séance
1. Valider scientifiquement deux hypothèses métier sur votre dataset à l'aide de **tests statistiques adaptés** (`scipy.stats`).
2. Construire une **segmentation / typologie non supervisée (K-Means)** combinant agrégation distribuée **PySpark**, transformation d'asymétrie (`log1p`), standardisation (`StandardScaler`) et labellisation métier vers **`/lakehouse/gold/`**.
3. Développer votre **Dashboard Streamlit interactif** dans `app/dashboard.py` (`http://localhost:8501`) en documentant chaque requête SQL et chaque choix statistique.

In [2]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("J4_Stats_Segmentation_Dashboard")
    .getOrCreate()
)

:: loading settings :: url = jar:file:/usr/local/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-e70873b8-8464-44f9-bd12-c7959b240256;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 281ms :: artifacts dl 27ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.1 from central in [default]
	io.delta#delta-storage;3.2.1 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |  

## 1. Inférence Statistique : Tester vos Hypothèses Métier (`scipy.stats`)

Choisissez **au moins deux tests statistiques** pertinents pour votre sujet parmi :
- **Comparaison de 2 groupes indépendants** sur une variable asymétrique : **Mann-Whitney U** (`stats.mannwhitneyu`).
- **Comparaison de $k \ge 3$ groupes** : **Kruskal-Wallis** (`stats.kruskal`).
- **Lien monotone entre 2 variables quantitatives** non gaussiennes : **Corrélation de rangs de Spearman** (`stats.spearmanr`).
- **Dépendance entre 2 variables qualitatives** : **Test du $\chi^2$** (`stats.chi2_contingency`).

### Exercice 1.1 — Formuler $H_0 / H_1$, projeter les colonnes utiles depuis Silver et conclure
> **Bonne pratique Big Data :** Ne faites jamais `.toPandas()` sur toute une table Silver brute ! Filtrez et sélectionnez (`select(...)`) uniquement les colonnes nécessaires côté PySpark avant de rapatrier les données vers le driver.

### Formulation des hypothèses

- **Test #1 — Kruskal-Wallis :**
  - **H0 :** la distribution de l'écart Elo entre l'équipe à domicile et l'équipe à l'extérieur est identique pour les victoires à domicile, les matchs nuls et les victoires à l'extérieur.
  - **H1 :** au moins un de ces groupes présente une distribution différente.
  - **Justification :** le test de Kruskal-Wallis permet de comparer plusieurs groupes indépendants sans supposer que les données suivent une loi normale.

- **Test #2 — Corrélation de Spearman :**
  - **H0 :** il n'existe pas de relation monotone entre l'écart Elo des équipes et l'écart de buts du match.
  - **H1 :** il existe une relation monotone entre l'écart Elo des équipes et l'écart de buts du match.
  - **Justification :** la corrélation de Spearman est adaptée pour mesurer une relation entre deux variables quantitatives sans supposer une distribution normale.

In [3]:
SILVER_MAIN_PATH = "/lakehouse/silver/matches"

# On ne récupère que les colonnes nécessaires aux tests statistiques
df_stats_spark = (
    spark.read
    .format("delta")
    .load(SILVER_MAIN_PATH)
    .select(
        "ftresult",
        "elo_difference",
        "goal_difference"
    )
    .filter(
        F.col("ftresult").isin("H", "D", "A")
        & F.col("elo_difference").isNotNull()
        & F.col("goal_difference").isNotNull()
    )
)

print("Nombre de matchs utilisés :", df_stats_spark.count())

# Conversion en Pandas uniquement après sélection des 3 colonnes utiles
df_sample = df_stats_spark.toPandas()


# ============================================================
# TEST 1 — KRUSKAL-WALLIS
# L'écart Elo est-il différent selon le résultat du match ?
# ============================================================

elo_home_win = df_sample.loc[
    df_sample["ftresult"] == "H",
    "elo_difference"
]

elo_draw = df_sample.loc[
    df_sample["ftresult"] == "D",
    "elo_difference"
]

elo_away_win = df_sample.loc[
    df_sample["ftresult"] == "A",
    "elo_difference"
]

stat_kruskal, p_kruskal = stats.kruskal(
    elo_home_win,
    elo_draw,
    elo_away_win
)

print("\n=== Test #1 : Kruskal-Wallis ===")
print(f"Statistique H : {stat_kruskal:.4f}")
print(f"p-value       : {p_kruskal:.6g}")

if p_kruskal < 0.05:
    print(
        "Conclusion : rejet de H0. "
        "La distribution de l'écart Elo diffère significativement "
        "selon le résultat du match."
    )
else:
    print(
        "Conclusion : H0 n'est pas rejetée. "
        "Aucune différence significative n'est mise en évidence."
    )


# ============================================================
# TEST 2 — SPEARMAN
# L'écart Elo est-il lié à l'écart de buts ?
# ============================================================

rho, p_spearman = stats.spearmanr(
    df_sample["elo_difference"],
    df_sample["goal_difference"]
)

print("\n=== Test #2 : Corrélation de Spearman ===")
print(f"Coefficient rho : {rho:.4f}")
print(f"p-value         : {p_spearman:.6g}")

if p_spearman < 0.05:
    print(
        "Conclusion : rejet de H0. "
        "Il existe une relation monotone significative "
        "entre l'écart Elo et l'écart de buts."
    )
else:
    print(
        "Conclusion : H0 n'est pas rejetée. "
        "Aucune relation monotone significative n'est mise en évidence."
    )

26/11/19 08:53:31 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Nombre de matchs utilisés : 140712

=== Test #1 : Kruskal-Wallis ===
Statistique H : 14120.5576
p-value       : 0
Conclusion : rejet de H0. La distribution de l'écart Elo diffère significativement selon le résultat du match.

=== Test #2 : Corrélation de Spearman ===
Coefficient rho : 0.3484
p-value         : 0
Conclusion : rejet de H0. Il existe une relation monotone significative entre l'écart Elo et l'écart de buts.


### Interprétation des tests statistiques

Les deux tests statistiques montrent des résultats significatifs.

Le test de Kruskal-Wallis donne une p-value extrêmement faible. Nous rejetons donc l'hypothèse H0. L'écart de niveau Elo entre les deux équipes n'est pas distribué de la même manière selon le résultat du match (victoire à domicile, match nul ou victoire à l'extérieur).

La corrélation de Spearman donne un coefficient de 0,3484 avec une p-value extrêmement faible. Nous rejetons également H0. Il existe donc une relation positive significative entre l'écart Elo et l'écart de buts.

Cette relation reste modérée, ce qui montre que le niveau Elo influence le résultat d'un match sans pour autant l'expliquer entièrement. D'autres facteurs interviennent donc également dans la performance des équipes.

## 2. Segmentation / Typologie par Clustering K-Means (`/lakehouse/gold/`)

Quelle que soit la thématique de votre dataset Kaggle, identifiez l'**entité métier pertinente à segmenter** (ex. clients/utilisateurs, stations, zones géographiques, produits, lignes de transport, capteurs...) :
1. **Agrégation PySpark** : résumez chaque entité par **3 à 5 features numériques comportementales** (fréquence, volume, valeur médiane/totale, récence, taux d'incident...).
2. **Prétraitement statistique** : appliquez `np.log1p(...)` sur les features positives asymétriques puis centrez-réduisez avec `StandardScaler()` afin qu'aucune variable à grande échelle n'écrase la distance euclidienne du K-Means.
3. **Interprétation métier** : inspectez les médianes par cluster (`groupby("cluster_id").median()`), attribuez un **nom explicite** à chaque segment (`colonne "segment"`), puis enregistrez la table dans **`/lakehouse/gold/<nom_table_segments>`**.

In [4]:
GOLD_SEGMENTS_PATH = "/lakehouse/gold/club_segments"

silver = (
    spark.read
    .format("delta")
    .load("/lakehouse/silver/matches")
)

# ============================================================
# 1. Transformer chaque match en performance par club
# ============================================================

home_teams = (
    silver
    .select(
        F.col("hometeam").alias("team"),
        F.col("homeelo").alias("elo"),
        F.col("fthome").alias("goals_scored"),
        F.col("ftaway").alias("goals_conceded"),
        F.when(F.col("ftresult") == "H", 1).otherwise(0).alias("win")
    )
)

away_teams = (
    silver
    .select(
        F.col("awayteam").alias("team"),
        F.col("awayelo").alias("elo"),
        F.col("ftaway").alias("goals_scored"),
        F.col("fthome").alias("goals_conceded"),
        F.when(F.col("ftresult") == "A", 1).otherwise(0).alias("win")
    )
)

team_matches = (
    home_teams
    .unionByName(away_teams)
    .filter(
        F.col("team").isNotNull()
        & F.col("elo").isNotNull()
        & F.col("goals_scored").isNotNull()
        & F.col("goals_conceded").isNotNull()
    )
)

# ============================================================
# 2. Agrégation au niveau du club
# ============================================================

club_features = (
    team_matches
    .groupBy("team")
    .agg(
        F.count("*").alias("nb_matches"),

        F.round(
            F.avg("elo"),
            2
        ).alias("avg_elo"),

        F.round(
            F.avg("goals_scored"),
            3
        ).alias("avg_goals_scored"),

        F.round(
            F.avg("goals_conceded"),
            3
        ).alias("avg_goals_conceded"),

        F.round(
            F.avg("win"),
            3
        ).alias("win_rate")
    )

    # On évite les clubs présents sur seulement quelques matchs
    .filter(
        F.col("nb_matches") >= 30
    )
)

print("Nombre de clubs conservés :", club_features.count())

club_features.orderBy(
    F.desc("nb_matches")
).show(20, truncate=False)

# ============================================================
# 3. Conversion Pandas seulement APRÈS agrégation
# ============================================================

entities_pd = club_features.toPandas()

print("\nDimensions du DataFrame :", entities_pd.shape)

entities_pd.describe()

Nombre de clubs conservés : 758


+--------------+----------+-------+----------------+------------------+--------+
|team          |nb_matches|avg_elo|avg_goals_scored|avg_goals_conceded|win_rate|
+--------------+----------+-------+----------------+------------------+--------+
|Norwich       |1139      |1534.59|1.356           |1.374             |0.384   |
|Birmingham    |1135      |1521.71|1.189           |1.275             |0.349   |
|Watford       |1130      |1537.76|1.327           |1.395             |0.35    |
|Derby         |1114      |1496.92|1.254           |1.313             |0.356   |
|Middlesbrough |1111      |1579.99|1.267           |1.194             |0.374   |
|Blackburn     |1102      |1566.09|1.29            |1.29              |0.363   |
|Burnley       |1099      |1565.26|1.241           |1.321             |0.359   |
|West Brom     |1091      |1591.48|1.289           |1.291             |0.359   |
|Wolves        |1083      |1583.31|1.283           |1.295             |0.377   |
|Crystal Palace|1079      |1

,nb_matches,avg_elo,avg_goals_scored,avg_goals_conceded,win_rate
count,758.000000,758.000000,758.000000,758.000000,758.000000
mean,395.067282,1467.155686,1.239040,1.408082,0.328616
std,334.405535,126.025569,0.282429,0.268294,0.103298
min,30.000000,1169.520000,0.212000,0.618000,0.061000
25%,86.000000,1390.535000,1.076250,1.244250,0.268000
50%,285.000000,1455.265000,1.194000,1.383000,0.320000
75%,695.750000,1529.922500,1.352750,1.540750,0.362000
max,1139.000000,1961.940000,2.506000,2.545000,0.748000


In [5]:
from sklearn.metrics import silhouette_score

# Variables utilisées pour segmenter les clubs
features = [
    "nb_matches",
    "avg_elo",
    "avg_goals_scored",
    "avg_goals_conceded",
    "win_rate"
]

X = entities_pd[features].copy()

# ============================================================
# 1. Correction de l'asymétrie
# ============================================================

# nb_matches est fortement asymétrique :
# on applique log1p uniquement sur cette variable
X["nb_matches"] = np.log1p(X["nb_matches"])

# ============================================================
# 2. Standardisation
# ============================================================

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Dimensions après préparation :", X_scaled.shape)

# ============================================================
# 3. Recherche du nombre de clusters
# ============================================================

results = []

for k in range(2, 7):

    kmeans_test = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    labels = kmeans_test.fit_predict(X_scaled)

    silhouette = silhouette_score(
        X_scaled,
        labels
    )

    results.append({
        "k": k,
        "inertia": kmeans_test.inertia_,
        "silhouette_score": silhouette
    })

k_selection = pd.DataFrame(results)

print("\nComparaison des nombres de clusters :")
display(k_selection)

Dimensions après préparation : (758, 5)

Comparaison des nombres de clusters :


,k,inertia,silhouette_score
0,2,2395.110581,0.315219
1,3,1722.773014,0.324112
2,4,1463.889033,0.244885
3,5,1296.136396,0.255769
4,6,1148.152132,0.255623


In [6]:
# ============================================================
# K-Means définitif avec k = 3
# ============================================================

kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=20
)

entities_pd["cluster_id"] = kmeans.fit_predict(X_scaled)

print("Répartition des clubs par cluster :")
print(
    entities_pd["cluster_id"]
    .value_counts()
    .sort_index()
)

# ============================================================
# Profil médian de chaque cluster
# ============================================================

cluster_profile = (
    entities_pd
    .groupby("cluster_id")[
        [
            "nb_matches",
            "avg_elo",
            "avg_goals_scored",
            "avg_goals_conceded",
            "win_rate"
        ]
    ]
    .median()
    .round(3)
)

print("\nProfil médian des clusters :")
display(cluster_profile)

Répartition des clubs par cluster :
cluster_id
0     72
1    300
2    386
Name: count, dtype: int64

Profil médian des clusters :


,nb_matches,avg_elo,avg_goals_scored,avg_goals_conceded,win_rate
cluster_id,,,,,
0,823.5,1638.985,1.840,1.007,0.560
1,82.0,1387.555,1.092,1.586,0.256
2,573.0,1493.870,1.229,1.316,0.338


In [7]:
# ============================================================
# Nom métier des clusters
# ============================================================

segment_mapping = {
    0: "Clubs dominants",
    1: "Clubs peu représentés / en difficulté",
    2: "Clubs intermédiaires établis"
}

entities_pd["segment"] = (
    entities_pd["cluster_id"]
    .map(segment_mapping)
)

assert "segment" in entities_pd.columns

print("Répartition des segments :")
print(entities_pd["segment"].value_counts())

display(
    entities_pd[
        [
            "team",
            "nb_matches",
            "avg_elo",
            "avg_goals_scored",
            "avg_goals_conceded",
            "win_rate",
            "cluster_id",
            "segment"
        ]
    ]
    .sort_values(
        ["cluster_id", "avg_elo"],
        ascending=[True, False]
    )
    .head(30)
)

# ============================================================
# Sauvegarde dans la couche Gold
# ============================================================

GOLD_SEGMENTS_PATH = "/lakehouse/gold/club_segments"

(
    spark.createDataFrame(entities_pd)
    .coalesce(2)
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_SEGMENTS_PATH)
)

print(
    f"\nTable de segmentation enregistrée dans "
    f"{GOLD_SEGMENTS_PATH} ({len(entities_pd):,} clubs)"
)

Répartition des segments :
segment
Clubs intermédiaires établis             386
Clubs peu représentés / en difficulté    300
Clubs dominants                           72
Name: count, dtype: int64


,team,nb_matches,avg_elo,avg_goals_scored,avg_goals_conceded,win_rate,cluster_id,segment
354,Barcelona,942,1961.94,2.366,0.916,0.674,0,Clubs dominants
214,Real Madrid,943,1959.08,2.291,0.989,0.676,0,Clubs dominants
558,Bayern Munich,869,1926.88,2.450,0.913,0.689,0,Clubs dominants
399,Man United,981,1884.94,1.814,1.002,0.585,0,Clubs dominants
639,Chelsea,980,1879.61,1.804,0.981,0.562,0,Clubs dominants
115,Arsenal,980,1878.04,1.896,1.023,0.576,0,Clubs dominants
585,Liverpool,981,1877.54,1.854,1.001,0.558,0,Clubs dominants
742,Juventus,961,1851.74,1.816,0.839,0.624,0,Clubs dominants
113,Man City,989,1842.58,1.936,1.047,0.564,0,Clubs dominants
122,Ath Madrid,947,1834.25,1.609,0.997,0.530,0,Clubs dominants



Table de segmentation enregistrée dans /lakehouse/gold/club_segments (758 clubs)


### Interprétation de la segmentation

Le K-Means a permis d'identifier trois profils de clubs.

Le premier groupe rassemble les clubs dominants, caractérisés par un Elo élevé, une forte capacité offensive, peu de buts encaissés et un taux de victoire important.

Le deuxième groupe contient des clubs moins présents dans le dataset et présentant globalement des performances plus faibles.

Le troisième groupe représente des clubs intermédiaires et bien établis, avec un nombre important de matchs mais des performances plus proches de la moyenne.

Cette segmentation permet de résumer les différents profils de clubs présents dans le dataset et pourra être utilisée dans le dashboard.

## 3. Livrable Final : Votre Application `app/dashboard.py` sur Streamlit (`:8501`)

Complétez maintenant **`app/dashboard.py`** pour valoriser vos tables `/lakehouse/gold/` :
- **Architecture respectée** : l'application Streamlit ne doit lire **que** des tables pré-calculées de la couche **Gold** (`/lakehouse/gold/...`), jamais les fichiers CSV bruts !
- **Interactivité** : au moins un filtre utilisateur (`st.selectbox`, `st.multiselect`, `st.slider`) et 3 à 4 visualisations Plotly Express complémentaires.
- **Rigueur explicative** : sous chaque visualisation clé, intégrez un `st.expander("Requête SQL & Justification statistique")` affichant la requête Spark SQL ayant produit la table Gold et justifiant le choix des indicateurs statistiques retenus.

In [ ]:
# Vérification finale de l'inventaire complet de votre couche Gold sur HDFS
jvm = spark._jvm
fs = jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())
PathHDFS = jvm.org.apache.hadoop.fs.Path
gold_root = PathHDFS("/lakehouse/gold")

gold_tables = [
    s.getPath().getName()
    for s in fs.listStatus(gold_root)
    if s.isDirectory() and fs.exists(PathHDFS(s.getPath(), "_delta_log"))
]
assert len(gold_tables) > 0, "Aucune table Delta trouvée dans /lakehouse/gold/ !"

print("=== Tables Delta disponibles dans /lakehouse/gold/ pour Streamlit ===")
for t in sorted(gold_tables):
    cnt = spark.read.format("delta").load(f"/lakehouse/gold/{t}").count()
    print(f" - /lakehouse/gold/{t:<25} : {cnt:>9,} lignes")

2026-11-19 09:04:58.360 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-19 09:04:58.362 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-19 09:04:58.513 
  command:

    streamlit run /usr/local/lib/python3.11/site-packages/ipykernel_launcher.py [ARGUMENTS]
2026-11-19 09:04:58.514 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-19 09:04:58.514 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-19 09:04:58.516 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-19 09:04:58.517 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running

DeltaGenerator()

## 4. Aperçu des visualisations finales

Les résultats produits dans les couches Gold sont également exploités dans le dashboard Streamlit interactif.

Les graphiques suivants permettent de visualiser les principaux résultats directement dans le notebook.

In [13]:
import plotly.express as px
import plotly.io as pio

# Thème clair pour avoir un rendu propre dans le notebook
pio.templates.default = "plotly_white"

In [14]:
# ============================================================
# Évolution annuelle du nombre moyen de buts par match
# ============================================================

yearly_pd = (
    spark.read
    .format("delta")
    .load("/lakehouse/gold/goals_evolution_yearly")
    .orderBy("year")
    .toPandas()
)

fig_year = px.line(
    yearly_pd,
    x="year",
    y="avg_goals_per_match",
    markers=True,
    title="Évolution du nombre moyen de buts par match",
    labels={
        "year": "Année",
        "avg_goals_per_match": "Buts moyens par match"
    },
    color_discrete_sequence=["#2563EB"]
)

fig_year.update_traces(
    line=dict(width=3),
    marker=dict(size=8)
)

fig_year.update_layout(
    title_x=0.5
)

fig_year.show()

In [15]:
# ============================================================
# Visualisation des segments K-Means
# ============================================================

segments_pd = (
    spark.read
    .format("delta")
    .load("/lakehouse/gold/club_segments")
    .toPandas()
)

segment_colors = {
    "Clubs dominants": "#16A34A",
    "Clubs intermédiaires établis": "#2563EB",
    "Clubs peu représentés / en difficulté": "#DC2626"
}

fig_clusters = px.scatter(
    segments_pd,
    x="avg_elo",
    y="win_rate",
    color="segment",
    size="nb_matches",
    hover_name="team",
    hover_data={
        "nb_matches": True,
        "avg_goals_scored": True,
        "avg_goals_conceded": True
    },
    title="Segmentation des clubs par K-Means",
    labels={
        "avg_elo": "Elo moyen",
        "win_rate": "Taux de victoire",
        "segment": "Segment",
        "nb_matches": "Nombre de matchs",
        "avg_goals_scored": "Buts marqués moyens",
        "avg_goals_conceded": "Buts encaissés moyens"
    },
    color_discrete_map=segment_colors
)

fig_clusters.update_layout(
    title_x=0.5
)

fig_clusters.show()

In [16]:
# ============================================================
# Top 15 des clubs ayant marqué le plus de buts
# ============================================================

goals_pd = (
    spark.read
    .format("delta")
    .load("/lakehouse/gold/club_goals_pareto")
    .orderBy("goals_rank")
    .limit(15)
    .toPandas()
)

goals_pd = goals_pd.sort_values(
    "total_goals",
    ascending=True
)

fig_goals = px.bar(
    goals_pd,
    x="total_goals",
    y="team",
    orientation="h",
    color="total_goals",
    text="total_goals",
    title="Top 15 des clubs ayant marqué le plus de buts",
    labels={
        "team": "Club",
        "total_goals": "Nombre total de buts"
    },
    color_continuous_scale="Blues"
)

fig_goals.update_traces(
    textposition="outside"
)

fig_goals.update_layout(
    title_x=0.5,
    coloraxis_showscale=False,
    height=600
)

fig_goals.show()